# Day 14 — Softmax properties: stability & differentiability

**Milestone 2:** two facts that make softmax *usable in practice*: it's **shift-invariant** (→ the subtract-the-max trick that prevents overflow) and **differentiable** (→ why attention can be trained at all).

## 1. Review (~5 min)

Recall **day 12** (exp) and **day 13** (softmax).

In [ ]:
import numpy as np
rng = np.random.default_rng(14)

**R1.** (day 12) elementwise $e^x$.

In [ ]:
def r_my_exp(x):
    raise NotImplementedError("# YOUR CODE HERE")

In [ ]:
def check_r1(fn):
    x = rng.normal(size=5)
    assert np.allclose(fn(x), np.exp(x))
    print("✅ Review R1 passed")

check_r1(r_my_exp)

**R2.** (day 13) softmax.

In [ ]:
def r_softmax(s):
    raise NotImplementedError("# YOUR CODE HERE")

In [ ]:
def check_r2(fn):
    s = rng.normal(size=6)
    e = np.exp(s - s.max()); oracle = e / e.sum()
    assert np.allclose(np.asarray(fn(s)), oracle)
    print("✅ Review R2 passed")

check_r2(r_softmax)

## 2. Lecture note (~10 min): why subtract the max, and why softmax is trainable

**The itch.** Two practical questions. (1) $e^{s}$ overflows to `inf` for large scores — how do we compute softmax safely? (2) Why use softmax at all instead of plain argmax?

**Property 1 — shift-invariance.** Adding a constant $c$ to every score leaves softmax unchanged:
$$\operatorname{softmax}(s + c)_i = \frac{e^{s_i}e^{c}}{\sum_j e^{s_j}e^{c}} = \operatorname{softmax}(s)_i,$$
because $e^{c}$ cancels top and bottom. So we may subtract $\max(s)$ first: the biggest input becomes 0, $e^0=1$, **nothing overflows**, and the answer is identical. Every real implementation does this.

**A useful cousin — log-sum-exp.** $\log\sum_i e^{s_i}$ shows up constantly (it's the log-normalizer of softmax). Computed naively it overflows too; the stable form is $\max(s) + \log\sum_i e^{\,s_i-\max(s)}$ — same subtract-the-max idea.

**Property 2 — differentiability.** Unlike argmax's jump, softmax is **smooth**: small changes in scores → small, well-defined changes in weights. That means gradients flow back through it, which is the entire reason attention can be *learned* end-to-end.

**Knobs & effect.** Subtract-the-max = the universal safety trick (free, exact). Differentiability = the property that makes the whole Transformer trainable.

**In the wild.** Numerically-stable softmax / cross-entropy / log-sum-exp in every deep-learning library.

## 3. Exercises (~15 min)

### Exercise 1 — stable softmax
Implement softmax by **subtracting the max before exponentiating**. (Checks: sums to 1, matches plain softmax on safe inputs, and stays finite on huge inputs like `[1000, 1001, 1002]`.)

In [ ]:
def softmax_stable(s):
    raise NotImplementedError("# YOUR CODE HERE")

In [ ]:
def check_e1(fn):
    s = rng.normal(size=6)
    e = np.exp(s - s.max()); oracle = e / e.sum()
    assert np.allclose(np.asarray(fn(s)), oracle)
    big = np.array([1000.0, 1001.0, 1002.0])
    out = np.asarray(fn(big))
    assert np.all(np.isfinite(out)) and np.allclose(out.sum(), 1.0)
    print("✅ Exercise 1 passed")

check_e1(softmax_stable)

### Exercise 2 — log-sum-exp, computed safely (the effect)
Return $\log\sum_i e^{s_i}$ using the stable form $\max(s) + \log\sum_i e^{\,s_i-\max(s)}$. (Checks: matches the naive value on safe inputs, stays finite on huge inputs where the naive version is `inf`.)

In [ ]:
def logsumexp(s):
    raise NotImplementedError("# YOUR CODE HERE")

In [ ]:
def check_e2(fn):
    s = rng.normal(size=6)
    assert np.allclose(fn(s), np.log(np.sum(np.exp(s))))
    big = np.array([1000.0, 1001.0, 1002.0])
    assert np.isfinite(fn(big))
    assert np.allclose(fn(big) - 1000.0, np.log(np.sum(np.exp(np.array([0.0, 1.0, 2.0])))))
    print("✅ Exercise 2 passed")

check_e2(logsumexp)

### Exercise 3 — shift-invariance (the property)
Return stable softmax, and the check confirms $\operatorname{softmax}(s)=\operatorname{softmax}(s+c)$ for random constants `c`.

In [ ]:
def softmax_shift(s):
    raise NotImplementedError("# YOUR CODE HERE")

In [ ]:
def check_e3(fn):
    for _ in range(5):
        s = rng.normal(size=7); c = rng.normal() * 50
        assert np.allclose(np.asarray(fn(s)), np.asarray(fn(s + c)))
    print("✅ Exercise 3 passed")

check_e3(softmax_shift)

## Solutions (collapsed — peek only if stuck)

`ref_`-prefixed answers. Running the next cell validates everything against the checks above.

In [ ]:
def ref_my_exp(x):
    return np.exp(x)

def ref_softmax(s):
    e = np.exp(s - np.max(s)); return e / e.sum()

def ref_softmax_stable(s):
    e = np.exp(s - np.max(s)); return e / e.sum()

def ref_logsumexp(s):
    m = np.max(s); return m + np.log(np.sum(np.exp(s - m)))

def ref_softmax_shift(s):
    return ref_softmax_stable(s)

check_r1(ref_my_exp)
check_r2(ref_softmax)
check_e1(ref_softmax_stable)
check_e2(ref_logsumexp)
check_e3(ref_softmax_shift)
print("\nAll reference solutions pass. 🎉  See you on day 15!")